# Soup Kernel Challenge — ID Automation (`IDA`)

**Decentrathon 6.0 · трек AI 1 «AI Engineering by Soup» · команда ID Automation (1 человек)**

Свои Triton-ядра (распаковка NF4 в обоих проходах, RMSNorm, пролог внимания + SDPA `is_causal`, кросс-энтропия с градиентом на месте, энтропия + argmax) и снятие лишних fp32-копий — без изменения задачи, данных, настроек и качества.
**Результаты этого прогона — в последней ячейке** (числа читаются из `runs/result.json` и `runs/checks.json`).
Методика — протокол жюри: два прогона базы и два прогона решения в одной сессии (база, решение, решение, база), в зачёт идут медианы.

## Как запускать

1. Справа: **Accelerator: GPU T4 x2**, **Internet: on**. Harness сам ограничивает замер одной картой (`CUDA_VISIBLE_DEVICES=0`), в этом ноутбуке ничего на GPU не запускается.
2. Если репозиторий приватный — Kaggle Secret **`GITHUB_TOKEN`** (Add-ons → Secrets) с правом только на чтение. Токен уходит в заголовок одной команды `git clone` и нигде не печатается и не сохраняется. Без секрета клон идёт анонимно (публичный репозиторий). Необязательный секрет `HF_TOKEN` — от лимитов Hugging Face.
3. **Run All**, без правок. Около **35 минут**: тесты, база и решение по 2 раза (база, решение, решение, база), оценка адаптеров, 10 проверок. Любая ошибка останавливает Run All.
4. Все числа в последней ячейке читаются только из `runs/result.json` и `runs/checks.json` этого прогона. Ничего не пишется в `/kaggle/input`; `harness/`, `tests/`, `task/` не меняются.

In [ ]:
REPO_URL = "https://github.com/danieliskrin30-lang/soup-kernel-v7.git"   # публичное зеркало того же коммита
# основной репозиторий платформы: https://github.com/BAITC-Hacks/hack-8539eaba-id-automation
COMMIT = "<COMMIT_SHA>"          # финальный коммит сдачи (полный SHA)

REPO = "/kaggle/working/repo"
assert COMMIT and not COMMIT.startswith("<"), "Впишите COMMIT — финальный коммит сдачи"

In [ ]:
import os, subprocess, sys
from importlib import metadata


def run(cmd, cwd=None, check=True):
    """Запуск с выводом построчно (он сохраняется в версии ноутбука); ошибка -> стоп Run All."""
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    out = []
    for line in p.stdout:
        print(line, end="")
        out.append(line)
    rc = p.wait()
    if check and rc:
        raise SystemExit(f"Команда завершилась с кодом {rc}: {' '.join(cmd)}")
    return rc, "".join(out)


gpus = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used",
                       "--format=csv,noheader"], capture_output=True, text=True).stdout
print(gpus)
assert "T4" in gpus, "Включите Accelerator: GPU T4 x2"

try:  # необязательно: токен Hugging Face от лимитов скачивания
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

run([sys.executable, "-m", "pip", "install", "-q", "soup-cli[train]==0.77.0",
     "transformers==5.16.1", "peft==0.20.0", "trl==0.29.1", "bitsandbytes==0.50.2",
     "accelerate==1.14.0", "pytest"])

print(f"python {sys.version.split()[0]}")
for pkg in ("soup-cli", "torch", "triton", "transformers", "peft", "trl", "bitsandbytes",
            "accelerate", "datasets", "pytest"):
    try:
        print(f"{pkg:13s} {metadata.version(pkg)}")
    except metadata.PackageNotFoundError:
        print(f"{pkg:13s} НЕ УСТАНОВЛЕН")

In [ ]:
import base64, shutil

token, basic = "", ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as exc:
    print("Секрета GITHUB_TOKEN нет, клонирую без него:", type(exc).__name__)
auth = []
if token:  # токен — только в заголовке этой команды: не в URL, не в файлах, не в выводе
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    auth = ["-c", f"http.extraHeader=Authorization: Basic {basic}"]

shutil.rmtree(REPO, ignore_errors=True)          # всегда свежая копия кода
clone = subprocess.run(["git", *auth, "clone", "-q", REPO_URL, REPO],
                       capture_output=True, text=True)
token, basic, auth = "", "", []                 # токен больше не нужен — стираем из памяти
if clone.returncode:
    raise SystemExit("git clone не удался: " + clone.stderr)
subprocess.run(["git", "-C", REPO, "checkout", "-q", COMMIT], check=True)

head = subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%H %s"],
                      capture_output=True, text=True).stdout.strip()
print("HEAD:", head)
assert head.startswith(COMMIT) or COMMIT.startswith(head.split()[0]), "HEAD не совпал с COMMIT"

sol = os.path.join(REPO, "solution")
files = sorted(f for f in os.listdir(sol) if os.path.isfile(os.path.join(sol, f)))
print(f"файлов в solution/: {len(files)}")
print(", ".join(files))
assert len(files) == 26, f"ожидалось 26 файлов в solution/, найдено {len(files)}"

In [ ]:
run([sys.executable, "task/prepare_data.py"], cwd=REPO)

In [ ]:
import re

rc, out = run([sys.executable, "-m", "pytest", "solution", "-q"], cwd=REPO, check=False)
summary = out.strip().splitlines()[-1] if out.strip() else ""
counts = {k: int(v) for v, k in re.findall(r"(\d+) (passed|failed|error|errors|skipped)", summary)}
PYTEST_SUMMARY = summary
print("\nИтог pytest:", summary)
if rc != 0 or counts.get("failed") or counts.get("error") or counts.get("errors") \
        or counts.get("passed") != 70:
    raise SystemExit(f"Тесты solution/ не прошли как ожидалось (нужно 70 passed): {summary}")

In [ ]:
# Протокол жюри: база, решение, решение, база; затем eval обоих адаптеров. ~30 минут.
run([sys.executable, "harness/run_task.py", "--repeats", "2", "--out", "runs"], cwd=REPO)

In [ ]:
rc, out = run([sys.executable, "harness/check_rules.py", "--result", "runs/result.json"],
              cwd=REPO, check=False)
assert "ALL CHECKS PASSED" in out, "check_rules.py: не все проверки пройдены (см. вывод выше)"
run([sys.executable, "harness/score.py", "runs/result.json"], cwd=REPO)

## Результаты этого прогона

**Результаты этого прогона — в последней ячейке:** speedup, отношение памяти, eval loss, время и пик каждого повтора, доля наших ядер (по прогонам и по ядрам), все 10 проверок и число тестов. Всё читается из `runs/result.json` и `runs/checks.json` этой версии ноутбука.

Методика — протокол жюри: два прогона базы и два прогона решения в одной сессии (база, решение, решение, база), в зачёт идут медианы. Конкретных чисел в тексте нет намеренно: на T4 время меняется от запуска к запуску, вывод ячейки всегда соответствует своему прогону.

In [ ]:
import json

res = json.load(open(os.path.join(REPO, "runs/result.json")))
chk = json.load(open(os.path.join(REPO, "runs/checks.json")))
s, r = res["summary"], res["summary"]["ratios"]

print("=== runs/result.json ===")
print(f"speedup        {r['speedup']:.4f}x   (время: база {s['base']['wall_s']} с -> решение {s['solution']['wall_s']} с, медианы)")
print(f"memory_ratio   {r['memory_ratio']:.4f}x   (пик: база {s['base']['gpu_peak_mib']} -> решение {s['solution']['gpu_peak_mib']} МиБ)")
print(f"eval loss      база {s['base']['eval_loss']} | решение {s['solution']['eval_loss']} | {r['quality_delta_pct']:+.3f}%")
print(f"цикл обучения  база {s['base']['train_loop_s']} с | решение {s['solution']['train_loop_s']} с (медианы)")

print("\nПо повторам:")
for side in ("base", "solution"):
    for run_ in res["runs"][side]:
        print(f"  {run_['run']:11s} wall {run_['wall_s']:7.2f} с   пик {run_['gpu_peak_mib']:6d} МиБ")

print("\nДоля наших ядер (без example_*, шаги 3+):")
for run_ in res["runs"]["solution"]:
    m = run_["metrics"]
    own = {k: v for k, v in (m.get("own_kernel_ms") or {}).items() if not k.startswith("example_")}
    share = sum(own.values()) / 1000 / m["timed_loop_s"]
    print(f"  {run_['run']:11s} {sum(own.values()):8.1f} мс / {m['timed_loop_s']:.2f} с = {100 * share:.2f}%")
    for k, v in sorted(own.items(), key=lambda kv: -kv[1]):
        print(f"      {k:32s} {v:9.1f} мс  {100 * v / 1000 / m['timed_loop_s']:5.2f}%")

print("\n=== runs/checks.json ===")
ok = sum(1 for c in chk["checks"] if c["ok"])
for c in chk["checks"]:
    print(f"{'PASS' if c['ok'] else 'FAIL'}  {c['check']}: {c['detail']}")
print(f"\nпройдено {ok}/{len(chk['checks'])}")
soup_tests = next((c["detail"] for c in chk["checks"] if "Soup regression" in c["check"]), "?")
print(f"тесты Soup (проверка №10): {soup_tests}")
print(f"наши тесты (pytest solution): {globals().get('PYTEST_SUMMARY', 'см. вывод ячейки с pytest')}")